# Chapter 2 — Token and Positional Embeddings

This notebook converts integer token IDs into the continuous vector representations consumed by a transformer. Tokenization gives every vocabulary item an index, but those indices are categorical labels rather than meaningful numerical measurements: token ID 100 is not “twice” token ID 50. An embedding layer solves this by using each ID as a lookup key for a trainable vector.

The notebook begins with a tiny controlled example so the lookup behavior can be inspected directly. A seeded `torch.nn.Embedding` layer creates a small weight matrix, and selecting one or several IDs retrieves the corresponding rows. The workflow then scales to the GPT-2 vocabulary size and reuses the data-loading pipeline defined in `bpe_tokenizer.ipynb` to obtain a real batch of token windows from `text-verdict.txt`.

Token embeddings alone identify *which* tokens are present but not *where* they occur. A second embedding layer assigns a vector to each position in the context window. Adding token and positional embeddings produces one tensor with vocabulary identity and sequence-order information. The preserved shapes trace that transformation from a batch of token IDs to model-ready vectors.

> The first cell uses `import_ipynb` and imports `create_dataloader` and `raw_text` from `bpe_tokenizer.ipynb`. Keep that source notebook and `text-verdict.txt` in the same folder when running this notebook.

## Learning objectives

By the end of the notebook, you should be able to:

- explain why token IDs require learned vector representations;
- interpret an embedding layer as a trainable lookup table;
- track batch, sequence, and embedding dimensions;
- create token embeddings for the GPT-2 vocabulary;
- create positional embeddings for a fixed context window;
- explain the broadcasting used when token and position vectors are added.

| Tensor | Shape in the preserved run | Meaning |
|---|---:|---|
| Token IDs | `[8, 4]` | Eight sequences of four tokens |
| Token embeddings | `[8, 4, 256]` | One 256-dimensional vector per token |
| Position embeddings | `[4, 256]` | One vector per context position |
| Combined embeddings | `[8, 4, 256]` | Token identity plus position |


## 1. Import PyTorch and reuse the chapter data pipeline

The captured setup cell imports PyTorch, `nn`, and `import_ipynb`, then loads `create_dataloader` and `raw_text` from the tokenizer notebook. `%%capture` suppresses import-time output so the notebook can focus on the embedding results.

This dependency connects the chapter stages directly:

```text
raw text → GPT-2 token IDs → batched token windows → embeddings
```


In [5]:
%%capture

import torch
from torch import nn
import import_ipynb
from bpe_tokenizer import create_dataloader, raw_text

## 2. Inspect an embedding lookup table

The small example uses six possible token IDs and three features per token. `nn.Embedding(6, 3)` therefore stores a trainable weight matrix:

$$
\mathbf{W}_{\text{embed}} \in \mathbb{R}^{6 \times 3}
$$

Setting the random seed makes the displayed weights reproducible for this saved run. Looking up token ID `3` returns row 3 of the matrix; passing several IDs returns the corresponding rows in the same order. No arithmetic relationship between the IDs is assumed.


In [17]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

In [18]:
torch.manual_seed(123)
embedding_layer = nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [19]:
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## 3. Scale token embeddings to the GPT-2 vocabulary

The next layer uses `vocab_size = 50257`, matching the GPT-2 tokenizer vocabulary, and `output_dim = 256`. Its weight matrix contains one trainable 256-dimensional vector for every possible token ID.

These vectors start from random values and would normally be learned jointly with the language model. This notebook creates and applies the layers but does not train them.


In [20]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = nn.Embedding(vocab_size, output_dim)

## 4. Create a batch of token windows

The reused data loader tokenizes `text-verdict.txt` and returns eight sequences with a context length of four. The preserved output shows an input tensor of shape `[8, 4]`, where the first dimension is batch size and the second is the number of token positions in each sequence.

The data loader also returns next-token targets, but only the input IDs are required for the embedding demonstration.


In [21]:
max_length = 4
dataloader = create_dataloader(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=True
)
data_iter = iter(dataloader)
inputs, outputs = next(data_iter)
print(f"Token IDs:\n {inputs}\n")
print(f"Input shape:\n {inputs.shape}\n")

Token IDs:
 tensor([[24818,   417,    12, 12239],
        [  314,  3114,   379,   262],
        [ 2156,   286,  4116,    13],
        [  866,   262,  2119,    11],
        [ 3363,    11,   340,   373],
        [  198,     1,    40,  2900],
        [  465, 14475,    13,   198],
        [ 3081,   286,  2045,  1190]])

Input shape:
 torch.Size([8, 4])



## 5. Convert token IDs into token embeddings

Passing the integer batch through `token_embedding_layer` replaces every scalar token ID with its 256-dimensional row from the embedding matrix. The tensor shape changes from `[batch, context]` to `[batch, context, embedding_dim]`.

The preserved result is `[8, 4, 256]`: eight sequences, four positions per sequence, and one vector of 256 features at each position.


In [22]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

## 6. Represent positions inside the context window

Token embeddings do not distinguish the first position from the fourth. A positional embedding layer creates one trainable vector for each index in the context window. `torch.arange(context_length)` supplies the position IDs `[0, 1, 2, 3]`.

With four positions and 256 features, the preserved positional tensor has shape `[4, 256]`. The same four position vectors can be reused across every sequence in the batch.


In [23]:
context_length = max_length
pos_embedding_layer = nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


## 7. Combine token identity and sequence position

The final representation is an element-wise sum:

$$
\mathbf{E}_{\text{input}} =
\mathbf{E}_{\text{token}} + \mathbf{E}_{\text{position}}
$$

PyTorch broadcasts the `[4, 256]` positional tensor across the batch dimension before adding it to the `[8, 4, 256]` token tensor. The resulting shape stays `[8, 4, 256]`.

Each vector now carries two signals: which token occupies the position and where that position lies in the current context window. These combined embeddings are the input representation passed to the attention layers developed later in the project.


In [24]:
# PyTorch broadcasts the position vectors across the batch dimension.
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Conclusion

This notebook completes the bridge from token IDs to transformer-ready vectors. The saved outputs make each dimensional change explicit and show how two embedding tables contribute complementary information.

| Stage | Transformation | Preserved shape |
|---|---|---:|
| Batched token IDs | Data loader output | `[8, 4]` |
| Token lookup | IDs → vocabulary vectors | `[8, 4, 256]` |
| Position lookup | Positions → context vectors | `[4, 256]` |
| Element-wise addition | Token + position | `[8, 4, 256]` |

No optimization occurs here, so the embedding values are initial parameters rather than learned semantic features. Later training will update them together with attention and feed-forward layers.


## Resources & References

- [Chapter 2 companion code — *Build a Large Language Model (From Scratch)*](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/01_main-chapter-code)
- [PyTorch `nn.Embedding` documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- [PyTorch data loading utilities](https://docs.pytorch.org/docs/stable/data.html)
- [Book page — *Build a Large Language Model (From Scratch)*](https://www.manning.com/books/build-a-large-language-model-from-scratch)
